# LLM-from-scratch-stack: End-to-End Mini GPT Tutorial

Welcome. This notebook walks through building a **small but real language model training pipeline** from raw data to instruction tuning.

You will:

1. Download a real dataset (WikiText-103)
2. Convert it into the JSONL format expected by the training stack
3. Train a Byte-Pair Encoding (BPE) tokenizer
4. Pretrain a small GPT model
5. Perform supervised fine-tuning (SFT) on instruction-style examples
6. Evaluate perplexity and run simple behavioral probes
7. Interactively generate text

This is not a frontier-scale model. It is a **clean, educational reference implementation** of the full LLM lifecycle.

---

## Why This Notebook Exists

Modern LLM workflows can feel opaque. Libraries hide the moving parts.

This tutorial intentionally exposes the core components:

* Tokenization
* Autoregressive training
* Loss masking
* Checkpointing
* Fine-tuning
* Sampling

The goal is clarity over scale.

---

## Architecture Overview

At a high level, the system looks like this:

```
Raw Text → JSONL → Tokenizer → GPT Pretraining → Instruction SFT → Generation
```

Each stage is implemented using the same modular training stack from the repository.

---

## What "From Scratch" Means Here

You are not using a pretrained HuggingFace model.

You are:

* Training your own tokenizer
* Initializing a GPT model from random weights
* Pretraining it on real text
* Fine-tuning it with structured instruction prompts
* Running inference manually with temperature + top-p sampling

That is the complete lifecycle of a modern LLM.

---

## Expected Behavior

After pretraining + SFT, the model should:

* Produce short coherent completions
* Answer small arithmetic prompts it saw during SFT
* Respond politely to greeting-style prompts
* Produce simple descriptive sentences

It will not:

* Be factually strong
* Perform complex reasoning
* Generalize far beyond its tiny dataset

This is a demonstration model, not a production system.

---

## Runtime Expectations

On GPU:

* Pretraining should take minutes depending on `max_steps`.
* SFT should complete quickly.

On CPU:

* Reduce step counts significantly.

---

## Recommended Usage

1. Restart kernel.
2. Run all cells.
3. Inspect the tokenizer artifact.
4. Watch the pretraining progress bar.
5. Observe loss curves.
6. Interact with the final model.

---

## What You Should Learn

By the end of this notebook, you should understand:

* How tokenization shapes model behavior
* Why autoregressive models predict next-token distributions
* How masking enables instruction tuning
* How sampling parameters affect output diversity
* Why data composition matters for behavior

---

Let’s begin.


In [1]:
# --- 0) Bootstrap: paths, imports, torch.load compat (PyTorch 2.6+) ---
from __future__ import annotations
import re
import json, os, random, sys
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from types import SimpleNamespace

import time
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import numpy as np

import numpy as np
import torch

def find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for _ in range(12):
        if (cur / "src" / "llmstack").exists() and (cur / "pyproject.toml").exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    # fallback: current working dir
    return Path.cwd().resolve()

REPO_ROOT = find_repo_root(Path.cwd())
#print("REPO_ROOT:", REPO_ROOT)

# ensure imports work
for p in [REPO_ROOT, REPO_ROOT/"src"]:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PRECISION = "bf16" if (DEVICE == "cuda" and torch.cuda.is_bf16_supported()) else ("fp16" if DEVICE == "cuda" else "fp32")
print("DEVICE:", DEVICE, "| PRECISION:", PRECISION)

DATA_DIR = REPO_ROOT / "data" / "toy"
ARTIFACTS_DIR = REPO_ROOT / "artifacts"
RUNS_DIR = REPO_ROOT / "runs"
for d in [DATA_DIR, ARTIFACTS_DIR, RUNS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def load_ckpt(path: Path, device):
    # PyTorch 2.6+ defaults weights_only=True; our checkpoints include more than raw tensors.
    return torch.load(path, map_location=device, weights_only=False)

if DEVICE == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True


CUDA available: True
GPU: NVIDIA GeForce RTX 4070 Laptop GPU
DEVICE: cuda | PRECISION: bf16


In [2]:
# --- 1) Repo imports ---
from llmstack.data.datamodule import DataModule
from llmstack.eval.perplexity import evaluate_perplexity
from llmstack.eval.probes import run_probes
from llmstack.model.gpt import GPTModel
from llmstack.optim.adamw import build_adamw
from llmstack.optim.schedulers import build_scheduler
from llmstack.train.engine import Trainer
from llmstack.utils.logging import RunLogger
from llmstack.utils.seed import seed_everything
from llmstack.tokenization.tokenizer import Tokenizer  # repo wrapper (loads tokenizer.json from a directory)

2026-02-19 20:17:49.083822: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-19 20:17:49.091369: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771550269.100903    9398 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771550269.104051    9398 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-02-19 20:17:49.114244: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

## 2) Download Dataset and Prepare Training Files

We now move from concept to data.

A language model learns by predicting the next token over massive text corpora. For this MVP, we use **WikiText-103**, a cleaned Wikipedia dataset commonly used in language modeling research.

We will:

1. Download the dataset using HuggingFace Datasets.
2. Extract the raw text field.
3. Convert it into JSONL format expected by the repo’s `DataModule`.

Why JSONL?

The training stack reads one example per line in the form:

```json
{"text": "some training string"}
```

This simple structure keeps the data loader modular and scalable.

We write two files:

* `data/toy/train.jsonl` → used for pretraining
* `data/toy/val.jsonl` → used for validation/perplexity evaluation

Each line is a single training example.

This step bridges external datasets and your internal training engine.

After this cell runs, you will have a clean, local corpus ready for tokenizer training and GPT pretraining.



In [4]:
SEED = 1337
seed_everything(SEED, deterministic=False)

# --- Download online dataset and write JSONL ---
try:
    from datasets import load_dataset
except ImportError:
    %pip -q install datasets
    from datasets import load_dataset

train_path = DATA_DIR / "train.jsonl"
val_path   = DATA_DIR / "val.jsonl"

MAX_TRAIN_LINES = None   # None = write ALL lines
MAX_VAL_LINES   = None   # None = write ALL lines

ds = load_dataset("wikitext", "wikitext-103-raw-v1")

def write_jsonl(split, out_path: Path, max_lines=None):
    n = 0
    with out_path.open("w", encoding="utf-8") as f:
        for item in split:
            text = (item.get("text") or "").strip()
            if not text:
                continue
            f.write(json.dumps({"text": text}, ensure_ascii=False) + "\n")
            n += 1
            if max_lines is not None and n >= max_lines:
                break
    return n

n_tr = write_jsonl(ds["train"], train_path, MAX_TRAIN_LINES)
n_va = write_jsonl(ds["validation"], val_path, MAX_VAL_LINES)

print(f"Wrote train lines: {n_tr} -> {train_path}")
print(f"Wrote val lines:   {n_va} -> {val_path}")

# --- Append a SMALL instruction-flavored set to pretraining ---
rng = random.Random(SEED)
instr_lines = []

# sample 1000 random additions
for _ in range(1000):
    a = rng.randrange(0, 101)
    b = rng.randrange(0, 101)
    instr_lines.append(f"Q: What is {a} + {b}?\n### Response:\nThe answer is {a+b}.")

# sample 500 random multiplications (smaller range)
for _ in range(500):
    a = rng.randrange(0, 21)
    b = rng.randrange(0, 21)
    instr_lines.append(f"Q: What is {a} * {b}?\n### Response:\nThe answer is {a*b}.")

with train_path.open("a", encoding="utf-8") as f:
    for line in instr_lines:
        f.write(json.dumps({"text": line}, ensure_ascii=False) + "\n")

print("Appended to pretraining:", len(instr_lines))


instr_lines += [
    "Q: Say hello politely.\n### Response:\nHello! Nice to meet you.",
    "Write one short sentence about the ocean.\nThe ocean is vast and constantly moving.",
    "Rewrite politely: Give me that.\nCould you please give me that?",
    "Continue: The city lights reflected on the wet street, and\n",
]

with train_path.open("a", encoding="utf-8") as f:
    for line in instr_lines:
        f.write(json.dumps({"text": line}, ensure_ascii=False) + "\n")

print("Appended instruction-format lines:", len(instr_lines))



Wrote train lines: 1165029 -> /home/jarettpoliner/Downloads/llm-from-scratch-stack-main/data/toy/train.jsonl
Wrote val lines:   2461 -> /home/jarettpoliner/Downloads/llm-from-scratch-stack-main/data/toy/val.jsonl
Appended to pretraining: 1500
Appended instruction-format lines: 1504


## 3) Train the Tokenizer (Creating `tokenizer.json`)

Before a model can learn language, it must learn how to break text into tokens.

A tokenizer defines the vocabulary and determines how raw strings are converted into integer IDs. Those IDs are what the GPT model actually sees.

In this stack, the `Tokenizer` wrapper expects a directory containing a trained tokenizer artifact:

```
artifacts/tokenizer/tokenizer.json
```

The tokenizer must include special tokens:

* `<pad>` → padding token
* `<bos>` → beginning-of-sequence token
* `<eos>` → end-of-sequence token
* `<unk>` → unknown token

We train a **byte-level BPE (Byte Pair Encoding)** tokenizer.

Why byte-level BPE?

* It avoids out-of-vocabulary issues.
* It works directly at the byte level.
* It handles arbitrary Unicode text robustly.
* It is widely used in modern LLMs.

Training the tokenizer consists of:

1. Scanning the training corpus.
2. Learning frequent byte merges.
3. Constructing a vocabulary of subword units.
4. Saving the resulting merge rules + vocab into `tokenizer.json`.

This file becomes a permanent artifact of the model.

Important: The model architecture depends on the tokenizer vocabulary size. Once trained, the tokenizer must remain fixed for the rest of pretraining and SFT.

After this step, you will have:

* A concrete vocabulary size.
* A consistent mapping from text → token IDs.
* A reusable artifact that makes the training stack deterministic and reproducible.

With the tokenizer in place, we are ready to initialize the GPT model.


In [6]:
# --- Train a BPE tokenizer and save as artifacts/tokenizer/tokenizer.json ---
try:
    from tokenizers import Tokenizer as HFTokenizer
    from tokenizers.models import BPE
    from tokenizers.pre_tokenizers import ByteLevel
    from tokenizers.trainers import BpeTrainer
except ImportError:
    %pip -q install tokenizers
    from tokenizers import Tokenizer as HFTokenizer
    from tokenizers.models import BPE
    from tokenizers.pre_tokenizers import ByteLevel
    from tokenizers.trainers import BpeTrainer

TOKENIZER_DIR = ARTIFACTS_DIR / "tokenizer"
TOKENIZER_DIR.mkdir(parents=True, exist_ok=True)
TOKENIZER_JSON = TOKENIZER_DIR / "tokenizer.json"

corpus_txt = ARTIFACTS_DIR / "tokenizer_corpus.txt"

def build_corpus_txt(jsonl_path: Path, out_txt: Path, text_field="text", max_lines=1_000_000):
    n = 0
    with jsonl_path.open("r", encoding="utf-8") as f_in, out_txt.open("w", encoding="utf-8") as f_out:
        for line in f_in:
            obj = json.loads(line)
            t = (obj.get(text_field) or "").strip()
            if not t:
                continue
            f_out.write(t.replace("\r\n", "\n") + "\n")
            n += 1
            if n >= max_lines:
                break
    return n

n = build_corpus_txt(train_path, corpus_txt)
print("Tokenizer corpus lines:", n)#, "->", corpus_txt)

VOCAB_SIZE = 8192  # better fluency for MVP  # MVP default; 1024 is too small for decent behavior
SPECIAL_TOKENS = ["<pad>", "<bos>", "<eos>", "<unk>"]

tok = HFTokenizer(BPE(unk_token="<unk>"))
tok.pre_tokenizer = ByteLevel(add_prefix_space=True)

trainer = BpeTrainer(vocab_size=VOCAB_SIZE, special_tokens=SPECIAL_TOKENS, min_frequency=2)
tok.train([str(corpus_txt)], trainer)
tok.save(str(TOKENIZER_JSON))

# Load via repo wrapper (expects artifact directory)
tokenizer = Tokenizer(TOKENIZER_DIR)
print("Tokenizer vocab_size:", tokenizer.vocab_size)#, "| bos:", tokenizer.bos_id, "| eos:", tokenizer.eos_id)


Tokenizer corpus lines: 1000000



Tokenizer vocab_size: 8192


## 4) Configure a Small GPT (Model + Training Settings)

Now we define the model itself.

A GPT model is controlled by two main groups of parameters:

1. **Architecture hyperparameters** (depth, width, sequence length)
2. **Training hyperparameters** (learning rate, batch size, number of steps)

For this notebook, we deliberately choose values that are:

* Small enough to run quickly on a single GPU
* Large enough to demonstrate meaningful learning
* Stable enough to train without extensive tuning

### Model Configuration

You are instantiating a compact decoder-only Transformer with:

* A fixed maximum sequence length
* A modest embedding dimension
* A small number of attention heads
* A small stack of transformer blocks

This keeps parameter count low while preserving the core mechanics of autoregressive language modeling.

### Training Knobs You Can Turn

Two parameters matter most for output quality:

* `PRETRAIN_STEPS` → how long the model sees raw text
* `MAX_TRAIN_LINES` → how much of the dataset you feed it

Increasing either improves quality — at the cost of runtime.

For a true MVP demonstration:

* A few thousand pretraining steps are enough to show structure.
* Tens of thousands begin producing noticeably better coherence.

### Mental Model

More data × more steps = better next-token prediction.

But this notebook prioritizes speed and clarity over scale.

Once configuration is set, we are ready to pretrain the model from random initialization.


In [9]:
def auto_batch_settings(seq_len: int):
    """Heuristic micro-batch + grad-accum for common VRAM sizes."""
    if DEVICE != "cuda":
        return 4, 1
    try:
        props = torch.cuda.get_device_properties(0)
        gb = props.total_memory / (1024**3)
    except Exception:
        gb = 16.0
    # conservative heuristics
    if gb <= 10:
        return 2, 4 if seq_len >= 512 else 2
    if gb <= 16:
        return 4, 2
    if gb <= 24:
        return 8, 1
    return 12, 1



# Training knobs (MVP)
PRETRAIN_STEPS = 30000  # increase for real GPU run      # increase to 3000-10000 for better results
SFT_STEPS = 5000           # stronger SFT so the instruction behavior shows up
RUN_DPO = False            # keep off for MVP

SEQ_LEN = 512  # reduce to 256 if VRAM is tight

AUTO_MICRO_BATCH, AUTO_ACCUM = auto_batch_settings(SEQ_LEN)
print(f"Auto batch: micro_batch={AUTO_MICRO_BATCH} grad_accum={AUTO_ACCUM}")


MICRO_BATCH = AUTO_MICRO_BATCH if DEVICE == "cuda" else 4
SFT_BATCH = 16 if DEVICE == "cuda" else 8


RUN_NAME = datetime.now().strftime("mvp_%Y%m%d_%H%M%S")
RUN_DIR = RUNS_DIR / RUN_NAME
CKPT_DIR = RUN_DIR / "checkpoints"
CKPT_DIR.mkdir(parents=True, exist_ok=True)

cfg = SimpleNamespace(
    model=SimpleNamespace(
        vocab_size=tokenizer.vocab_size,
        n_layers=8,
        n_heads=8,
        d_model=512,
        d_ff=2048,
        max_seq_len=SEQ_LEN,
        dropout=0.1,
        norm_type="layernorm",
        rope_enabled=True,
        tie_embeddings=True,
        gradient_checkpointing=False,
    ),
    data=SimpleNamespace(
        train_path=str(train_path),
        val_path=str(val_path),
        format="jsonl",
        text_field="text",
        seq_len=SEQ_LEN,
        pack_sequences=True,
        num_workers=0,
        shuffle=True,
        streaming=False,
        add_bos=True,
        add_eos=True,
        pad_to_seq_len=True,
    ),
    train=SimpleNamespace(
        seed=SEED,
        device=DEVICE,
        precision=PRECISION,
        compile=True,
        grad_clip=1.0,
        grad_accum_steps=AUTO_ACCUM,
        micro_batch_size=MICRO_BATCH,
        max_steps=PRETRAIN_STEPS,
        eval_interval=max(100, PRETRAIN_STEPS // 5),
        log_interval=max(20, PRETRAIN_STEPS // 25),
        save_interval=max(200, PRETRAIN_STEPS // 2),
        out_dir=str(RUNS_DIR),
        run_name=RUN_NAME,
        resume_path=None,
        max_eval_batches=50,
        deterministic=False,
    ),
    optim=SimpleNamespace(
        name="adamw",
        lr=2e-4,
        betas=(0.9, 0.95),
        weight_decay=0.1,
        eps=1e-8,
    ),
    sched=SimpleNamespace(name="warmup_cosine", warmup_steps=1000, min_lr=2e-5),
    log=SimpleNamespace(use_wandb=False, wandb_project="llmstack", jsonl=True, tensorboard=True),
    run_dir=str(CKPT_DIR),
)

#print("RUN_DIR:", RUN_DIR)


Auto batch: micro_batch=2 grad_accum=4


## 5) Pretraining

We now train the GPT model from random initialization.

Pretraining teaches the model to predict the **next token** given all previous tokens. This is autoregressive language modeling:

```
P(x_t | x_1, ..., x_{t-1})
```

For example:

```
The ocean is vast and → deep / blue / mysterious
```

The model learns this pattern across every token in the dataset.

---

### What Happens Each Step

1. Sample a batch of token sequences.
2. Run them through the Transformer.
3. Compute cross-entropy next-token loss.
4. Backpropagate.
5. Update weights.

This repeats for `PRETRAIN_STEPS` iterations.

---

### What Pretraining Gives You

* Grammar and syntax
* Basic word associations
* Statistical structure of language

It does not yet teach instruction following. That happens during SFT.

---

### What to Monitor

* Training loss should decrease.
* Validation perplexity should improve.

After pretraining, you have a GPT checkpoint that models raw language patterns.

Next: instruction fine-tuning.



In [ ]:
dm = DataModule(cfg, tokenizer, world_size=1, rank=0)

model = GPTModel(cfg.model).to(DEVICE)
optimizer = build_adamw(model, cfg.optim)
scheduler = build_scheduler(optimizer, cfg.sched, cfg.train.max_steps)

logger = RunLogger(str(RUN_DIR), jsonl=cfg.log.jsonl, tensorboard=cfg.log.tensorboard)

trainer = Trainer(cfg, model, optimizer, scheduler, logger, dm.train_dataloader(), dm.val_dataloader(), device=DEVICE)
trainer.fit()
trainer.save_checkpoint("pretrain_last.pt")
logger.close()

PRETRAIN_CKPT = CKPT_DIR / "pretrain_last.pt"
assert PRETRAIN_CKPT.exists(), PRETRAIN_CKPT
print("Saved pretrain checkpoint:", PRETRAIN_CKPT)

Pretraining:   0%|          | 0/30000 [00:00<?, ?it/s]

## 6) Evaluation Utilities

Before moving to fine-tuning, we evaluate the pretrained model.

Evaluation answers a simple question: **Is the model learning meaningful structure?**

We use two lightweight metrics:

### 1) Perplexity

Perplexity measures how well the model predicts the next token on held-out validation data.

Lower perplexity = better next-token prediction.

It is the standard metric for autoregressive language modeling.

---

### 2) Simple Behavioral Probes

We also run small prompt-based probes to observe qualitative behavior.

These are not formal benchmarks. They provide intuition about:

* Coherence
* Repetition
* Basic pattern learning

---

Evaluation serves two purposes:

* Verify that training is working
* Establish a baseline before instruction fine-tuning

Next, we shift from modeling raw language to modeling structured instruction-response behavior.



In [ ]:
@torch.no_grad()
def run_eval(ckpt_path: Path, label: str) -> dict:
    eval_model = GPTModel(cfg.model).to(DEVICE)
    ckpt = load_ckpt(ckpt_path, DEVICE)
    eval_model.load_state_dict(ckpt["model"])
    eval_model.eval()

    ppl = evaluate_perplexity(eval_model, dm.val_dataloader(), DEVICE, max_batches=cfg.train.max_eval_batches)
    probes = run_probes(eval_model, tokenizer, DEVICE)
    report = {"label": label, "checkpoint": str(ckpt_path), "perplexity": ppl, "probes": probes}

    out = RUN_DIR / f"eval_{label}.json"
    out.write_text(json.dumps(report, indent=2), encoding="utf-8")
    print(json.dumps(report, indent=2))
    return report

pretrain_report = run_eval(PRETRAIN_CKPT, "pretrain")

## 7) Supervised Fine-Tuning (SFT)

Pretraining teaches the model general language structure. SFT teaches it **how to respond**.

Here we fine-tune using instruction–response pairs.

### Key Implementation Details

To remain consistent with pretraining:

* Prepend `<bos>`
* Append `<eos>`
* Mask loss over the prompt region

Only the response tokens contribute to the loss. The model learns:

```
Prompt → Desired Response
```

---

### Why Mask the Prompt?

During SFT, we do not want the model to relearn the prompt. We want it to learn how to *continue* the prompt correctly.

Masking ensures gradients flow only through response tokens.

---

### Why Longer Responses?

Using structured responses like:

```
The answer is 9.
```

provides stronger supervision than single-token outputs.

This stabilizes learning and improves instruction-style behavior.

After SFT, the model should follow basic prompts rather than merely continue raw text.


In [ ]:
import re
import random
import torch

def randomize_plus_spacing(prompt: str, rng: random.Random) -> str:
    def repl(m):
        a, b = m.group(1), m.group(2)
        left  = " " if rng.random() < 0.5 else ""
        right = " " if rng.random() < 0.5 else ""
        return f"{a}{left}+{right}{b}"
    return re.sub(r"(\d+)\s*\+\s*(\d+)", repl, prompt)
# -------------------------------------------------
# SFT step (response-only loss with BOS/EOS)
# -------------------------------------------------
def sft_step_bos_eos(model, tokenizer, batch, separator: str, device: str, rng=None):
    ids_list = []
    labels_list = []

    pad_id = getattr(tokenizer, "pad_id", 0)
    bos_id = tokenizer.bos_id
    eos_id = tokenizer.eos_id

    for ex in batch:
        prompt = ex["prompt"]
        resp = ex["response"]

        # optional: randomize "a+b" spacing variants per batch
        if rng is not None:
            prompt = randomize_plus_spacing(prompt, rng)

        full = prompt + separator + resp

        ids_full = [bos_id] + tokenizer.encode(full) + [eos_id]
        ids_cut  = [bos_id] + tokenizer.encode(prompt + separator)
        cut = len(ids_cut)

        # response-only labels
        lab = ids_full.copy()
        lab[:cut] = [-100] * cut

        ids_list.append(ids_full)
        labels_list.append(lab)

    # pad to a single tensor for one forward pass (GPU efficient)
    max_len = max(len(x) for x in ids_list)
    B = len(ids_list)

    x = torch.full((B, max_len), pad_id, dtype=torch.long, device=device)
    y = torch.full((B, max_len), -100, dtype=torch.long, device=device)

    for i, (ids, lab) in enumerate(zip(ids_list, labels_list)):
        L = len(ids)
        x[i, :L] = torch.tensor(ids, dtype=torch.long, device=device)
        y[i, :L] = torch.tensor(lab, dtype=torch.long, device=device)

    _, loss = model(x, labels=y)
    return loss

# -------------------------------------------------
# Build SFT pools (balanced)
# -------------------------------------------------
rng = random.Random(SEED)

# --- SFT arithmetic pool ---
arith_examples = []

# additions 0..100
for a in range(0, 101):
    for b in range(0, 101):
        arith_examples.append({
            "prompt": f"Q: What is {a} + {b}?\n### Response:\n",
            "response": f"The answer is {a+b}."
        })

# multiplications 0..20
for a in range(0, 21):
    for b in range(0, 21):
        arith_examples.append({
            "prompt": f"Q: What is {a} * {b}?\n### Response:\n",
            "response": f"The answer is {a*b}."
        })

print("arith_examples:", len(arith_examples))
# Instruction pool
instr_examples = []

topics = [
    "the ocean", "a forest", "New York City", "machine learning",
    "a sunrise", "a marathon", "coffee", "music",
    "space exploration", "winter"
]

for t in topics:
    instr_examples.append({
        "prompt": f"Write one short sentence about {t}.\n",
        "response": f"{t.capitalize()} is fascinating in its own way."
    })
    instr_examples.append({
        "prompt": f"Write two short sentences about {t}.\n",
        "response": f"{t.capitalize()} can be beautiful. It can also be powerful."
    })

pairs = [
    ("Say hello politely.", "Hello! Nice to meet you."),
    ("Ask for help politely.", "Could you please help me with this?"),
    ("Decline politely.", "Thanks for asking, but I’ll have to pass."),
    ("Thank someone politely.", "Thank you, I really appreciate it."),
]
for p, r in pairs:
    instr_examples.append({"prompt": f"Q: {p}\n### Response:\n", "response": r})

qa = [
    ("Opposite of cold?", "Hot."),
    ("Color of grass?", "Green."),
    ("What do bees make?", "Honey."),
    ("What planet do we live on?", "Earth."),
]
for q, a in qa:
    instr_examples.append({"prompt": f"Q: {q}\n### Response:\n", "response": a})

# Identity / reasoning additions
instr_examples += [
    {"prompt": "Q: What is your name?\n### Response:\n",
     "response": "I am a small language model trained for demonstration purposes."},

    {"prompt": "Q: How old are you?\n### Response:\n",
     "response": "I do not have an age, but I was recently trained."},

    {"prompt": "Q: Why do you think so?\n### Response:\n",
     "response": "Because it depends on the context and information available."},

    {"prompt": "Explain in one sentence what machine learning is.\n",
     "response": "Machine learning is a method for teaching computers to recognize patterns from data."},
]

# Augment
aug = []
for ex in instr_examples:
    aug.append(ex)
    aug.append({"prompt": ex["prompt"].replace("short", "brief"),
                "response": ex["response"]})
instr_examples = aug

print("Arithmetic examples:", len(arith_examples))
print("Instruction examples:", len(instr_examples))


# -------------------------------------------------
# Load pretrained model
# -------------------------------------------------
sft_model = GPTModel(cfg.model).to(DEVICE)
sft_model.load_state_dict(load_ckpt(PRETRAIN_CKPT, DEVICE)["model"])
sft_model.train()

sft_optim = torch.optim.AdamW(sft_model.parameters(), lr=5e-5)

separator = ""  # prompts already include response header

SFT_BATCH = 16 if DEVICE == "cuda" else 8
ARITH_FRAC = 0.2


def sample_batch(arith_pool, instr_pool, batch_size, arith_frac, rng):
    n_arith = int(round(batch_size * arith_frac))
    n_instr = batch_size - n_arith
    batch = []
    for _ in range(n_arith):
        batch.append(arith_pool[rng.randrange(len(arith_pool))])
    for _ in range(n_instr):
        batch.append(instr_pool[rng.randrange(len(instr_pool))])
    rng.shuffle(batch)
    return batch


# -------------------------------------------------
# SFT Training Loop with Timer + Progress Bar
# -------------------------------------------------
loss_history = []
start_time = time.time()

progress = tqdm(range(SFT_STEPS), desc="SFT Training")

for step in progress:
    batch = sample_batch(arith_examples, instr_examples,
                         SFT_BATCH, ARITH_FRAC, rng)

    loss = sft_step_bos_eos(
        sft_model,
        tokenizer,
        batch,
        separator=separator,
        device=DEVICE,
        rng=rng,   # pass rng to enable randomization
    )

    sft_optim.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(sft_model.parameters(), 1.0)
    sft_optim.step()

    loss_val = loss.item()
    loss_history.append(loss_val)

    progress.set_postfix(loss=f"{loss_val:.4f}")

end_time = time.time()

print(f"\nSFT training completed in {(end_time - start_time):.2f} seconds")


# -------------------------------------------------
# Save Checkpoint
# -------------------------------------------------
SFT_CKPT = CKPT_DIR / "sft_last.pt"
torch.save({"model": sft_model.state_dict(), "step": SFT_STEPS}, SFT_CKPT)
print("Saved SFT checkpoint:", SFT_CKPT)


# -------------------------------------------------
# Plot Loss Curve
# -------------------------------------------------
plt.figure(figsize=(8, 4))
plt.plot(loss_history)
plt.yscale("log")
plt.title("SFT Training Loss")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.grid(True)
plt.show()

# Optional smoothed curve
if len(loss_history) > 50:
    smoothed = np.convolve(loss_history,
                           np.ones(50)/50,
                           mode='valid')
    plt.figure(figsize=(8, 4))
    plt.plot(smoothed)
    plt.yscale("log")
    plt.title("Smoothed SFT Loss")
    plt.xlabel("Step")
    plt.ylabel("Loss")
    plt.grid(True)
    plt.show()


In [ ]:
sft_report = run_eval(SFT_CKPT, "sft")

## 8) Text Generation

With pretraining and SFT complete, we can generate text.

Generation works by repeatedly:

1. Feeding the current token sequence into the model.
2. Extracting the final-step logits.
3. Sampling the next token.
4. Appending it to the sequence.

This continues until `<eos>` is produced or `max_new_tokens` is reached.

---

### Key Implementation Details

Several small details matter for correct behavior:

* `GPTModel` returns `(logits, loss)` → use only the logits.
* Use `tokenizer.eos_id` to stop generation.
* Decode only newly generated tokens (not the prompt).
* Crop the input context to `max_seq_len` to prevent overflow.

---

### Sampling Controls

Generation quality depends heavily on sampling:

* **Temperature** → controls randomness.
* **Top-p (nucleus sampling)** → restricts sampling to the most probable mass.
* **Repetition penalty** → discourages loops.

These do not change the model’s knowledge. They only change how we sample from its distribution.

After this step, you can interact with your trained mini-LLM and observe its learned behavior.



In [ ]:
import torch.nn.functional as F

def sample_top_p(probs: torch.Tensor, p: float):
    sorted_probs, sorted_idx = torch.sort(probs, descending=True)
    cdf = torch.cumsum(sorted_probs, dim=-1)
    mask = cdf > p
    mask[..., 0] = False
    sorted_probs[mask] = 0
    sorted_probs = sorted_probs / sorted_probs.sum().clamp(min=1e-8)
    pick = torch.multinomial(sorted_probs, 1)
    return sorted_idx[pick]

import re
import torch.nn.functional as F

@torch.no_grad()
def generate_text(
    model,
    tokenizer,
    prompt: str,
    max_new_tokens: int = 64,
    temperature: float = 0.2,
    top_p: float = 0.9,
    repetition_penalty: float = 1.1,
):
    model.eval()
    prompt_ids = [tokenizer.bos_id] + tokenizer.encode(prompt)
    x = torch.tensor([prompt_ids], device=DEVICE, dtype=torch.long)

    for _ in range(max_new_tokens):

        if x.size(1) > cfg.model.max_seq_len:
            x = x[:, -cfg.model.max_seq_len:]

        logits, _ = model(x)
        next_logits = logits[:, -1, :].squeeze(0)

        # repetition penalty
        if repetition_penalty and repetition_penalty != 1.0:
            for t in set(x[0].tolist()):
                if next_logits[t] > 0:
                    next_logits[t] /= repetition_penalty
                else:
                    next_logits[t] *= repetition_penalty

        next_logits = next_logits / max(temperature, 1e-6)
        probs = F.softmax(next_logits, dim=-1)

        if top_p is not None:
            sorted_probs, sorted_idx = torch.sort(probs, descending=True)
            cdf = torch.cumsum(sorted_probs, dim=-1)
            mask = cdf > top_p
            mask[..., 0] = False
            sorted_probs[mask] = 0
            sorted_probs = sorted_probs / sorted_probs.sum().clamp(min=1e-8)
            next_id = sorted_idx[torch.multinomial(sorted_probs, 1)]
        else:
            next_id = torch.multinomial(probs, 1)

        x = torch.cat([x, next_id.view(1, 1)], dim=1)

        if next_id.item() == tokenizer.eos_id:
            break

    # decode continuation only
    gen_ids = x[0].tolist()[len(prompt_ids):]
    text = tokenizer.decode(gen_ids)

    # cleanup BPE artifacts
    text = text.replace("Ġ", " ")
    text = " ".join(text.split())

    return text.strip()



# Load SFT model
infer = GPTModel(cfg.model).to(DEVICE)
infer.load_state_dict(load_ckpt(SFT_CKPT, DEVICE)["model"])
infer.eval()

tests = [
    "Q: Say hello politely.\n### Response:\n",
    "Write one short sentence about the ocean.\n",
    "Q: What is 2 + 7?\n### Response:\n",
]


for t in tests:
    print("Prompt:", t)
    print("Generated:", generate_text(infer, tokenizer, t, max_new_tokens=48))
    print()


## 8) Talk to it youself!

In [ ]:
# --- Interactive Prompt Loop ---

@torch.no_grad()
def chat_loop(model, tokenizer):
    model.eval()
    print("\nLLM ready. Type 'exit' to quit.\n")

    while True:
        user_input = input("You: ").strip()
        if user_input.lower() in {"exit", "quit"}:
            print("Exiting.")
            break

        # Match SFT format
        if not user_input.endswith("\n"):
            user_input += "\n"

        # If it looks like a question, use Q/Response format
        if user_input.lower().startswith("q:"):
            prompt = user_input + "### Response:\n"
        elif user_input.endswith("?"):
            prompt = f"Q: {user_input}\n### Response:\n"
        else:
            prompt = user_input

        output = generate_text(
            model,
            tokenizer,
            prompt,
            max_new_tokens=80,
            temperature=0.1,   # lower = more stable
            top_p=1.0,
            repetition_penalty=1.05,
        )

        print("LLM:", output)
        print()

# Load trained SFT checkpoint
infer = GPTModel(cfg.model).to(DEVICE)
infer.load_state_dict(load_ckpt(SFT_CKPT, DEVICE)["model"])
infer.eval()

chat_loop(infer, tokenizer)
